<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_33_rest_api_design/note_lesson_33_rest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌦️ Урок 33 — REST: дизайн API на метеоданих

В уроці 32 ми були **клієнтом** чужого API. Сьогодні — інша роль: ми **проєктуємо** API. Приклад — справжній проєкт викладача: сервіс бере синоптичні телеграми метеостанцій України з ogimet.com, розкодовує їх у температуру, тиск і вітер і віддає через API. Перша версія API працює, але користуватися нею важко: дієслова в адресах, «помилка» з кодом 200. Розберемо, якими бувають API взагалі, і як версія 2 виправляє першу за правилами REST.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | Meteo API v2 у ноутбуці, телеграма SYNOP | 0.1 | |
| 1 | які бувають API: CSV, JSON-RPC, GraphQL, SOAP, gRPC, WebSocket, SSE, webhook | 1.1–1.13 | |
| 2 | REST: ресурси, методи, OpenAPI, кейс v1 → v2 | 2.1–2.3 | |
| 3 | статус-коди і формат помилок | 3.1–3.4 | |
| 4 | `PATCH` і `DELETE` | 4.1–4.3 | |
| 5 | фільтри, вибір полів, пагінація | 5.1–5.5 | **Задача 1** |
| 6 | довга операція: `202 Accepted` | 6.1–6.2 | |
| 7 | клієнт до API і аналіз у pandas | 7.1–7.4 | **Задача 2** |
| 8 | той самий API без мережі: `TestClient` | 8.1 | |
| 9 | знайди помилку: 200 з `{"error": …}` | 9.1 | **Задача 3** |
| 10 | зупиняємо сервери | 10.1 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**: приклади змінюють дані на сервері, і наступні на це розраховують. Клітинка з визначенням (`def`, класи, константи) лише створює об'єкт — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Інтернет не потрібен: сервер, дані й усі клієнти працюють у ядрі ноутбука (у Colab інтернет потрібен лише першій клітинці — завантажити папку уроку й бібліотеки).
- Теорія, схеми й таблиці вибору типу API — у книзі: [Урок 33](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_33/).

---
## 0. Meteo API v2

Проєкт уроку — пакет `meteo_api` (FastAPI) і клієнт `weather_map/client.py` поруч із цим ноутбуком. У Colab ноутбук відкривається **без** файлів репозиторію, тому клітинка нижче завантажує лише папку уроку з GitHub (`git clone --sparse`) і ставить бібліотеки, яких бракує.

In [ ]:
import importlib.util
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_33_rest_api_design"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("meteo_api"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

packages = {"fastapi": "fastapi", "uvicorn": "uvicorn", "pymetdecoder": "pymetdecoder",
            "strawberry": "strawberry-graphql", "httpx": "httpx", "websockets": "websockets",
            "grpc": "grpcio", "grpc_tools": "grpcio-tools", "pandas": "pandas"}
missing = [pip_name for module, pip_name in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), "weather_map"))
print("папка уроку:", os.getcwd())

In [ ]:
import asyncio
import hashlib
import hmac
import json
import socket
import threading
import time
import xml.etree.ElementTree as ET
from http.server import BaseHTTPRequestHandler, HTTPServer

import httpx
import requests
import websockets

import meteo_api.app as meteo_app
from meteo_api.app import app, start_server
from meteo_api.decoder import decode_synop


def pick_port(preferred):
    """Порт для сервера: той, що в книзі, а якщо він зайнятий — будь-який вільний."""
    with socket.socket() as probe:
        try:
            probe.bind(("127.0.0.1", preferred))
        except OSError:
            probe.bind(("127.0.0.1", 0))
        return probe.getsockname()[1]

Сервер запускаємо у фоновому потоці (uvicorn). Книга показує порт 8032; якщо він зайнятий, `pick_port` візьме вільний, тому всі приклади беруть адресу зі змінної `BASE`.

In [ ]:
if meteo_app._server is None:
    BASE = start_server(pick_port(8032))
else:                                     # клітинку запустили вдруге — сервер уже працює
    BASE = f"http://127.0.0.1:{meteo_app._server.config.port}"
print(BASE)
print(requests.get(f"{BASE}/api/v1/stations", params={"limit": 3}, timeout=5).json())

У сховищі поки одна справжня станція — 34504 (Дніпро) з одним спостереженням. Щоб приклади не залежали від дня, коли зроблено знімок даних, додамо **умовну станцію** `99001` «Навчальна» не через API (у нашому API немає `POST /stations`), а прямо в сховище, і вісім **умовних** спостережень — уже через API (значення вигадані).

In [ ]:
repo = app.state.repo
if "99001" not in {station["wmo"] for station in repo.list_stations()}:
    repo.add_station("99001", "Навчальна")
OBS = f"{BASE}/api/v1/stations/99001/observations"
for hour, temperature in zip(range(0, 24, 3), [11.2, 10.4, 12.9, 17.5, 20.1, 19.3, 15.8, 13.0]):
    response = requests.post(OBS, json={"time": f"2026-09-20T{hour:02d}:00Z", "temperature": temperature,
                                        "pressure": 1005.0 + hour / 3}, timeout=5)
    print(response.status_code, response.headers.get("Location", response.json()))

### Приклад 0.1. Що всередині телеграми

Телеграма SYNOP — рядок груп по п'ять цифр. Розкодовує її бібліотека `pymetdecoder`; обгортка з проєкту — `meteo_api/decoder.py`.

In [ ]:
telegram = "AAXX 02181 34504 32975 51106 10251 20129 39989 40151 52027 80001 333 10330"
for key, value in decode_synop(telegram).items():
    print(f"{key:19} {value}")

`10251` — група температури: `1` — ознака групи, `0` — плюс, `251` → 25.1 °C; `39989` — тиск на станції 998.9 гПа. Відносну вологість телеграма не передає — `decoder.py` обчислює її з температури й точки роси. API віддає саме ці розкодовані поля.

---
## 1. Які бувають API

**API** — домовленість, як одна програма просить іншу щось зробити. Наш сервер віддає ті самі метеодані через різні типи API одразу:

```text
тип            хто починає        формат             адреса
HTTP + CSV     клієнт             CSV-текст          GET /cgi-bin/getsynop
REST           клієнт             JSON               /api/v1/...
JSON-RPC       клієнт             JSON               POST /rpc
GraphQL        клієнт             JSON + мова        POST /graphql
SOAP           клієнт             XML-конверт        POST /soap
gRPC           клієнт             Protobuf (двійк.)  окремий сервер, HTTP/2
WebSocket      будь-хто           JSON по каналу     /ws/observations
SSE            сервер (потік)     текстові події     GET /events/observations
webhook        сервер дзвонить    JSON               POST на адресу клієнта
```

### Приклад 1.1. HTTP + CSV: так працює ogimet

In [ ]:
params = {"block": "34504", "begin": "202409020000", "end": "202409022359"}
response = requests.get(f"{BASE}/cgi-bin/getsynop", params=params, timeout=5)
print(response.status_code, response.headers["Content-Type"])
print(response.text)

Це теж API: адреса, параметри, формат відповіді. Але назв полів немає — клієнт має знати формат рядка напам'ять.

### Приклад 1.2. Розбираємо рядок CSV самі

Ось що доводиться писати кожному клієнту такого API: позиція в рядку = значення.

In [ ]:
def parse_getsynop_line(line):
    station, year, month, day, hour, minute, telegram = line.split(",", 6)
    return {"station": station, "time": f"{year}-{month}-{day}T{hour}:{minute}Z", "telegram": telegram.rstrip("=")}

In [ ]:
for line in response.text.splitlines():
    print(parse_getsynop_line(line))

empty = requests.get(f"{BASE}/cgi-bin/getsynop", params={**params, "block": "00000"}, timeout=5)
print(empty.status_code, repr(empty.text))

Для неіснуючої станції — `200` і порожня відповідь: клієнт не відрізнить «станції немає» від «за цю добу немає телеграм». REST-версія скаже `404`.

### Приклад 1.3. JSON-RPC: назва методу — в тілі

**RPC** — «виклич функцію на іншому комп'ютері». Один URL, а метод і аргументи — у тілі ([JSON-RPC 2.0](https://www.jsonrpc.org/specification)).

In [ ]:
call = {"jsonrpc": "2.0", "method": "temperature.latest", "params": {"wmo": "34504"}, "id": 1}
print(requests.post(f"{BASE}/rpc", json=call, timeout=5).json())

call = {"jsonrpc": "2.0", "method": "temperature.forecast", "params": {"wmo": "34504"}, "id": 2}
wrong = requests.post(f"{BASE}/rpc", json=call, timeout=5)
print(wrong.status_code, wrong.json())

Навіть помилка — `200 OK`: у RPC помилка живе в полі `error` з кодом зі стандарту (`-32601` — «метод не існує»), а не в статусі HTTP.

### Приклад 1.4. Маленький RPC-клієнт

In [ ]:
class RpcError(Exception):
    pass


def rpc_call(method, call_id=1, **params):
    """Викликати метод JSON-RPC і повернути result або кинути RpcError."""
    reply = requests.post(f"{BASE}/rpc", json={"jsonrpc": "2.0", "method": method, "params": params, "id": call_id},
                          timeout=5).json()
    if "error" in reply:
        raise RpcError(f"{reply['error']['code']}: {reply['error']['message']}")
    return reply["result"]

In [ ]:
print("станцій:", rpc_call("stations.count"))
print(rpc_call("station.get", wmo="99001"))
for method, kwargs in [("station.get", {"wmo": "00000"}), ("station.get", {"code": "34504"})]:
    try:
        rpc_call(method, **kwargs)
    except RpcError as error:
        print("RpcError", error)

Код `404` тут — номер помилки всередині JSON-RPC (сервер обрав його сам), а `-32602` — стандартне «неправильні параметри». Клієнт виглядає як виклик звичайних функцій — у цьому сила RPC. Але кеші, проксі й браузер не знають, що `station.get` лише читає дані: для них це просто `POST`.

### Приклад 1.5. GraphQL: клієнт сам обирає поля

In [ ]:
query = """
{
  station(wmo: "34504") {
    name
    observations(last: 1) { time temperature }
  }
}
"""
print(requests.post(f"{BASE}/graphql", json={"query": query}, timeout=5).json())

Відповідь повторює форму запиту: `station → name, observations → time, temperature`. Тиску немає — ми його не просили. У REST знадобилося б два запити: станція, потім спостереження.

### Приклад 1.6. GraphQL: назва станції і два останні спостереження з тиском

In [ ]:
query = '{ station(wmo: "99001") { name observations(last: 2) { time pressure } } }'
data = requests.post(f"{BASE}/graphql", json={"query": query}, timeout=5).json()["data"]
print(data)
assert data["station"]["name"] == "Навчальна"
assert data["station"]["observations"] == [{"time": "2026-09-20T18:00Z", "pressure": 1011.0},
                                           {"time": "2026-09-20T21:00Z", "pressure": 1012.0}]

### Приклад 1.7. GraphQL: помилка теж приходить з кодом 200

In [ ]:
import logging

logging.getLogger("strawberry.execution").setLevel(logging.CRITICAL)   # сервер не друкує помилку в журнал ноутбука
bad = requests.post(f"{BASE}/graphql", json={"query": '{ station(wmo: "34504") { name colour } }'}, timeout=5)
print(bad.status_code)
print(bad.json()["errors"][0]["message"])

missing = requests.post(f"{BASE}/graphql", json={"query": '{ station(wmo: "00000") { name } }'}, timeout=5)
print(missing.status_code, missing.json())

GraphQL перевіряє запит за **схемою**: поля `colour` у типу `Station` немає — помилка в полі `errors`. Неіснуюча станція — `null`, а не 404. Тому клієнти GraphQL перевіряють `errors` і `null`, а не статус HTTP.

### Приклад 1.8. SOAP: XML-конверт

In [ ]:
envelope = """<?xml version="1.0" encoding="utf-8"?>
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/" xmlns:m="urn:meteo">
  <soap:Body>
    <m:GetLatestTemperature><m:Station>34504</m:Station></m:GetLatestTemperature>
  </soap:Body>
</soap:Envelope>"""
soap_response = requests.post(f"{BASE}/soap", data=envelope.encode(), timeout=5,
                              headers={"Content-Type": "text/xml; charset=utf-8"})
print(soap_response.status_code)
print(soap_response.text)

### Приклад 1.9. Дістаємо значення з SOAP-відповіді

XML розбираємо `xml.etree.ElementTree`; теги мають простори імен, тому шукаємо `{urn:meteo}Temperature`.

In [ ]:
NS = {"soap": "http://schemas.xmlsoap.org/soap/envelope/", "m": "urn:meteo"}


def soap_latest_temperature(xml_text):
    root = ET.fromstring(xml_text)
    result = root.find("soap:Body/m:GetLatestTemperatureResponse", NS)
    if result is None:
        raise ValueError(root.findtext("soap:Body/soap:Fault/faultstring", namespaces=NS))
    return result.findtext("m:Time", namespaces=NS), float(result.findtext("m:Temperature", namespaces=NS))

In [ ]:
print(soap_latest_temperature(soap_response.text))
fault = requests.post(f"{BASE}/soap", data=envelope.replace("34504", "00000").encode(), timeout=5)
print(fault.status_code)
try:
    soap_latest_temperature(fault.text)
except ValueError as error:
    print("SOAP Fault:", error)
print(len(envelope.encode()), "байтів запиту SOAP проти", len(json.dumps(call).encode()), "у JSON-RPC")

Той самий виклик «дай температуру», що в JSON-RPC, але в кілька разів довший. На практиці SOAP-клієнт генерують з WSDL бібліотекою [`zeep`](https://docs.python-zeep.org/).

### Приклад 1.10. gRPC: двійковий контракт

Контракт — файл `meteo_api/meteo.proto`; з нього `grpcio-tools` генерує класи для сервера й клієнта. gRPC працює окремим сервером поверх HTTP/2.

In [ ]:
try:
    import grpc

    import meteo_api.grpc_meteo as grpc_meteo
except ImportError as error:
    grpc = None
    print("⚠️ gRPC пропущено — немає бібліотеки:", error.name)
else:
    if "GRPC_TARGET" not in globals():           # повторний запуск клітинки не стартує другий сервер
        GRPC_TARGET = grpc_meteo.start_grpc_server(port=pick_port(50051))
    stub, pb2 = grpc_meteo.connect(GRPC_TARGET)
    reply = stub.GetLatest(pb2.StationRequest(wmo="34504"), timeout=5)
    print(type(reply).__name__, reply.station, reply.time, reply.temperature)
    print(len(reply.SerializeToString()), "байтів у двійковому вигляді")
    try:
        stub.GetLatest(pb2.StationRequest(wmo="99999"), timeout=5)
    except grpc.RpcError as error:
        print(error.code(), error.details())

Повідомлення двійкові — коротші за JSON, але людині не прочитати. Помилки — власні коди gRPC (`NOT_FOUND`), а не HTTP-статуси. З браузера gRPC напряму не викликати, тому його обирають для зв'язку **сервісів між собою**.

### Приклад 1.11. WebSocket: сервер шле сам

Постійний двосторонній канал: клієнт підписується на станцію, сервер надсилає спостереження одне за одним. У Jupyter корутину запускаємо через `await`.

In [ ]:
async def listen(wmo, limit=4):
    received = []
    async with websockets.connect(BASE.replace("http", "ws") + "/ws/observations") as ws:
        await ws.send(json.dumps({"subscribe": wmo}))
        async for message in ws:
            received.append(json.loads(message))
            if len(received) == limit:
                break
    return received

In [ ]:
for message in await listen("99001"):
    print("отримано:", message)

Ми вийшли з циклу після чотирьох повідомлень — з'єднання закрилося, сервер перестав надсилати. Так працюють чати й котирування (урок 46).

### Приклад 1.12. SSE: потік подій у звичайній HTTP-відповіді

Server-Sent Events простіші: відповідь типу `text/event-stream`, яку сервер не закриває й дописує подія за подією. Так ChatGPT і Claude показують відповідь по словах.

In [ ]:
with httpx.stream("GET", f"{BASE}/events/observations", params={"station": "34504"}, timeout=5) as stream:
    print(stream.headers["content-type"])
    for line in stream.iter_lines():
        print(repr(line))

### Приклад 1.13. Webhook: «не дзвоніть нам — ми подзвонимо»

Клієнт один раз реєструє **свою** адресу, а сервер сам робить `POST` на неї, коли стається подія. Піднімемо маленький «приймач» на вільному порту; захист від підробок — підпис HMAC-SHA256 у заголовку `X-Meteo-Signature` (урок 17).

In [ ]:
received = []


class Receiver(BaseHTTPRequestHandler):
    def do_POST(self):
        body = self.rfile.read(int(self.headers["Content-Length"]))
        received.append((self.headers["X-Meteo-Signature"], body))
        self.send_response(204)
        self.end_headers()

    def log_message(self, *args):
        pass

In [ ]:
from meteo_api.api_types import WEBHOOK_SECRET

receiver = HTTPServer(("127.0.0.1", pick_port(8099)), Receiver)
threading.Thread(target=receiver.serve_forever, daemon=True).start()
hook_url = f"http://127.0.0.1:{receiver.server_address[1]}/meteo"

print(requests.post(f"{BASE}/webhooks", json={"url": hook_url}, timeout=5).json())
print(requests.post(f"{BASE}/webhooks/test-event", json={"station": "34504"}, timeout=5).json())
signature, body = received[0]
print(body.decode())

expected = hmac.new(WEBHOOK_SECRET, body, hashlib.sha256).hexdigest()
print("підпис справжній:", hmac.compare_digest(signature, expected))
print("підробка пройде:", hmac.compare_digest(signature, hmac.new(b"guess", body, hashlib.sha256).hexdigest()))

receiver.shutdown()
receiver.server_close()

`hmac.compare_digest`, а не `==`: звичайне порівняння зупиняється на першій різниці, і за часом відповіді можна підбирати підпис посимвольно. Приймач ми зупинили одразу після прикладу.

Для публічного API і для більшості вебзастосунків вибір за замовчуванням — **REST**. Решта уроку — про нього.

---
## 2. REST: ресурси, методи, контракт

```text
/api/v1
├── /stations                                    колекція станцій
│   └── /stations/34504                          одна станція
│       └── /stations/34504/observations         колекція спостережень
│           └── …/observations/2024-09-02T18:00Z одне спостереження
├── /observations/latest                         останні по всіх станціях
└── /imports                                     задачі імпорту
```

Адреси — **іменники в множині**, ідентифікатор — у шляху, фільтри — в параметрах, дія — **метод HTTP**, версія — на початку (`/api/v1`).

### Приклад 2.1. Станція і одне спостереження

In [ ]:
print(requests.get(f"{BASE}/api/v1/stations/34504", timeout=5).json())
obs = requests.get(f"{BASE}/api/v1/stations/34504/observations/2024-09-02T18:00Z", timeout=5).json()
print(obs["temperature"], obs["pressure"], obs["wind_speed"], obs["relative_humidity"])

Ідентифікатор спостереження читається: `2024-09-02T18:00Z` (ISO 8601), а не склеєне `3450420249218`.

### Приклад 2.2. OpenAPI: контракт, який пише сам код

FastAPI будує опис API за стандартом **OpenAPI** з коду: шляхи, методи, моделі. На ньому працює `/docs` (Swagger UI).

In [ ]:
spec = requests.get(f"{BASE}/openapi.json", timeout=5).json()
print(spec["info"]["title"], spec["info"]["version"])
for path, methods in spec["paths"].items():
    if path.startswith("/api/v1"):
        print(f"{' '.join(method.upper() for method in methods):17} {path}")

### Приклад 2.3. Кейс v1 → v2: та сама дія — нова адреса

Адреси першої версії API (файл `meteo_api/legacy/main.py`) і їхні REST-відповідники. Виконаємо лише читання — `DELETE` справжнього спостереження нам ще знадобиться.

In [ ]:
V2 = {
    "POST /filter_telegrams/": "GET /api/v1/stations/34504/observations?hour=18",
    "POST /download_telegrams": "POST /api/v1/imports",
    "GET /telegram/ua/3450420249218": "GET /api/v1/stations/34504/observations/2024-09-02T18:00Z",
    "DELETE /telegram/ua/3450420249218": "DELETE /api/v1/stations/34504/observations/2024-09-02T18:00Z",
}

In [ ]:
for old, new in V2.items():
    method, path = new.split(" ", 1)
    note = ""
    if method == "GET":
        note = f"→ {requests.get(BASE + path, timeout=5).status_code}"
    print(f"{old:36} ⇒ {new} {note}")

| Було (v1) | Проблема | Стало (v2) |
|---|---|---|
| `POST /filter_telegrams/` | дієслово в URL; `POST` для читання — не кешується | `GET …/observations?hour=18` |
| `POST /download_telegrams` — чекає хвилини | клієнт висить до тайм-ауту | `POST /api/v1/imports` → `202` (розділ 6) |
| `/telegram/ua/3450420249218` | склеєний id, назва колекції бази в URL | `…/observations/2024-09-02T18:00Z` |
| «даних немає» з кодом `200` | `raise_for_status()` мовчить | `404` + `{"detail": …}` |

**🔮 Прогноз:** Що має повернути `POST /api/v1/imports`, якщо імпорт триває хвилини?

<details>
<summary>Відповідь</summary>

`202 Accepted` і заголовок `Location` з адресою задачі; клієнт потім перевіряє її стан через `GET`. Перевіримо в розділі 6.

</details>

---
## 3. Статус-коди і формат помилок

Код відповіді — перше, що дивиться клієнт (`raise_for_status` з уроку 32). Тому він має казати правду.

### Приклад 3.1. Створено, вже є, погані дані, не знайдено

Чотири запити до станції 99001: створити спостереження, те саме вдруге, температура 75 °C, неіснуюча станція.

In [ ]:
body = {"time": "2026-09-21T00:00Z", "temperature": 9.5}
created = requests.post(OBS, json=body, timeout=5)
again = requests.post(OBS, json=body, timeout=5)
too_hot = requests.post(OBS, json={"time": "2026-09-21T03:00Z", "temperature": 75}, timeout=5)
missing = requests.get(f"{BASE}/api/v1/stations/00000", timeout=5)

print(created.status_code, created.headers["Location"])
print(again.status_code, again.json())
print(too_hot.status_code, too_hot.json()["detail"][0]["msg"])
print(missing.status_code, missing.json())
codes = [r.status_code for r in (created, again, too_hot, missing)]
assert codes == [201, 409, 422, 404], codes

- `201 Created` + `Location` — адреса нового ресурсу: клієнту не треба її вгадувати.
- `409 Conflict` — спостереження на цей строк уже є; повторний `POST` не створив дубль.
- `422` — тіло не пройшло перевірку моделі Pydantic (урок 37).
- `404` — станції немає. Тіло помилки завжди однакової форми `{"detail": …}`.

### Приклад 3.2. Що всередині 422

FastAPI пояснює **яке** поле і **чому** — клієнт може показати це користувачу.

In [ ]:
broken = requests.post(OBS, json={"time": "вчора", "temperature": "тепло", "wind_speed": -3}, timeout=5)
print(broken.status_code)
for problem in broken.json()["detail"]:
    print(problem["loc"], "—", problem["msg"])

Одна відповідь — усі три проблеми: неправильний час, температура не число, від'ємна швидкість вітру.

### Приклад 3.3. Який код обрати: правило як функція

Порядок перевірок сервера: хто ти → чи маєш право → чи існує → чи правильні дані → чи не суперечить стану → що зроблено.

In [ ]:
def choose_status(known=True, allowed=True, exists=True, valid=True, conflict=False, action="read"):
    if not known:
        return 401
    if not allowed:
        return 403
    if not exists:
        return 404
    if not valid:
        return 422
    if conflict:
        return 409
    return {"read": 200, "update": 200, "create": 201, "delete": 204, "accepted": 202}[action]

In [ ]:
cases = {
    "без токена": dict(known=False),
    "чужа станція": dict(allowed=False),
    "станції 00000 немає": dict(exists=False),
    "температура 75 °C": dict(valid=False),
    "строк уже є": dict(conflict=True, action="create"),
    "створено": dict(action="create"),
    "видалено": dict(action="delete"),
    "імпорт прийнято": dict(action="accepted"),
}
for situation, kwargs in cases.items():
    print(f"{situation:20} → {choose_status(**kwargs)}")

### Приклад 3.4. Методу немає — `405 Method Not Allowed`

`PUT` у нашому API не реалізовано. Сервер не каже «не знайдено» — ресурс є, а метод не підтримується. Заголовок `Allow` підказує дозволений метод (FastAPI кладе туди методи першого маршруту з цією адресою — тут `GET`); повний список видно в OpenAPI (приклад 2.2).

In [ ]:
put = requests.put(f"{OBS}/2026-09-20T12:00Z", json={"temperature": 20.4}, timeout=5)
print(put.status_code, put.json(), "| Allow:", put.headers.get("allow"))

---
## 4. PATCH і DELETE

`PATCH` змінює **лише передані поля**; `PUT` замінює ресурс **цілком**. Для виправлення однієї температури правильний метод — `PATCH`.

### Приклад 4.1. Виправляємо температуру

In [ ]:
url = f"{OBS}/2026-09-20T12:00Z"
fixed = requests.patch(url, json={"temperature": 20.4}, timeout=5).json()
print(fixed["time"], fixed["temperature"], fixed["pressure"])
assert fixed["temperature"] == 20.4 and fixed["pressure"] == 1009.0

Тиск `1009.0` лишився: `PATCH` торкнувся лише температури.

### Приклад 4.2. PATCH з поганими даними і для неіснуючого строку

In [ ]:
print(requests.patch(url, json={"temperature": -300}, timeout=5).status_code)
print(requests.patch(f"{OBS}/2026-09-20T13:00Z", json={"temperature": 1}, timeout=5).json())
print(requests.get(url, timeout=5).json()["temperature"], "— не змінилась")

`422` — дані неправильні, і нічого не змінилось. `404` — спостереження на 13:00 немає (строки кожні 3 години).

### Приклад 4.3. DELETE двічі

In [ ]:
url = f"{OBS}/2026-09-21T00:00Z"
first, second = requests.delete(url, timeout=5), requests.delete(url, timeout=5)
print(first.status_code, repr(first.text), "|", second.status_code, second.json())
print("GET після:", requests.get(url, timeout=5).status_code)

`204 No Content` — видалено, тіла немає. Другий раз — `404`.

**🔮 Прогноз:** Коди двох `DELETE` різні. Чому `DELETE` все одно називають ідемпотентним?

<details>
<summary>Відповідь</summary>

Ідемпотентність — про **стан сервера**, а не про код відповіді: після одного чи двох запитів спостереження однаково немає.

</details>

---
## 5. Фільтри, поля і пагінація

12 станцій × 8 строків × 365 днів — понад 35 тисяч спостережень за рік. Віддавати все одним шматком не можна, тому колекції віддають **сторінками**, а клієнт просить **лише потрібні поля**.

### Приклад 5.1. Лише потрібні поля

In [ ]:
rows = requests.get(OBS, params={"fields": "time,temperature", "limit": 100}, timeout=5).json()["items"]
print(len(rows), rows[:2])
assert all(set(row) == {"time", "temperature"} for row in rows)
assert rows[4] == {"time": "2026-09-20T12:00Z", "temperature": 20.4}
full = requests.get(OBS, params={"limit": 1}, timeout=5).json()["items"][0]
print(len(full), "полів без fields:", list(full))

### Приклад 5.2. Невідоме поле — помилка клієнта, а не мовчання

In [ ]:
response = requests.get(OBS, params={"fields": "time,colour"}, timeout=5)
print(response.status_code, response.json())

Якби сервер мовчки ігнорував `colour`, клієнт з одруківкою (`temprature`) отримував би порожні дані й не знав чому.

### Приклад 5.3. Фільтри: строк і проміжок часу

In [ ]:
noon = requests.get(OBS, params={"hour": 12, "fields": "time,temperature"}, timeout=5).json()
print("hour=12:", noon["items"])

evening = requests.get(OBS, params={"date_from": "2026-09-20T15:00Z", "date_to": "2026-09-20T21:00Z",
                                    "fields": "time"}, timeout=5).json()
print("15:00–21:00:", [item["time"][11:16] for item in evening["items"]])
print(requests.get(OBS, params={"hour": 25}, timeout=5).status_code, "— hour=25 не пройшов перевірку")

Фільтрує **сервер**: клієнт отримує лише потрібне, а межі (`0 ≤ hour ≤ 23`) перевіряються так само, як тіло POST.

### Приклад 5.4. Конверт сторінки

In [ ]:
page = requests.get(OBS, params={"fields": "time,temperature", "limit": 3}, timeout=5).json()
print("total:", page["total"], "| limit:", page["limit"], "| offset:", page["offset"])
for item in page["items"]:
    print(item)
print("next:", page["next"])

`items` — дані, `total` — скільки всього, `next` — **готове посилання** на наступну сторінку або `None`, якщо сторінка остання. Клієнт не рахує `offset` сам — сервер може змінити спосіб пагінації, не зламавши клієнтів.

### Приклад 5.5. Гортаємо за посиланнями `next`

In [ ]:
url, params = OBS, {"fields": "time", "limit": 3}
while url:
    page = requests.get(url, params=params, timeout=5).json()
    print(page["offset"], [item["time"][11:16] for item in page["items"]], "next:", page["next"] is not None)
    url, params = page["next"], None          # у next параметри вже є

**🔮 Прогноз:** Скільки запитів знадобиться з `limit=4`? А з `limit=100`?

<details>
<summary>Відповідь</summary>

8 спостережень: з `limit=4` — два (4 + 4; на другій сторінці `next` уже `None`), з `limit=100` — один.

</details>

### Задача 1. `fetch_all(url, limit)`

Напиши функцію, яка проходить **усі** сторінки колекції за посиланнями `next` і повертає пару `(усі елементи списком, скільки запитів зроблено)`.

- перший запит — `GET url` з параметром `limit`;
- наступні — за готовим посиланням `next` (без своїх параметрів);
- зупинись, коли `next` стане `None`.

Перевірка — у наступній клітинці.

In [ ]:
def fetch_all(url, limit):
    # BEGIN SOLUTION
    items, requests_made, params = [], 0, {"limit": limit}
    while url:
        page = requests.get(url, params=params, timeout=5).json()
        requests_made += 1
        items += page["items"]
        url, params = page["next"], None
    return items, requests_made
    # END SOLUTION

In [ ]:
total = requests.get(OBS, params={"limit": 1}, timeout=5).json()["total"]
result = fetch_all(OBS, limit=3)
assert result is not None, "функція має повертати пару (елементи, кількість запитів)"
items, made = result
assert len(items) == total, f"очікували {total} елементів, отримали {len(items)}"
assert made == -(-total // 3), f"для {total} елементів по 3 потрібно {-(-total // 3)} запити, зроблено {made}"
assert [item["time"] for item in items] == sorted(item["time"] for item in items)
items, made = fetch_all(OBS, limit=100)
assert len(items) == total and made == 1
print(total, "спостережень")
print("✅ Задача 1 розв'язана")

`offset` / `limit` — найпростіша пагінація. Великі API (GitHub, Stripe) використовують **курсор** — «дай 100 записів після запису X»: це стабільніше, коли нові записи з'являються посеред гортання.

---
## 6. Довга операція: 202 Accepted

У першій версії `POST /download_telegrams` завантажував телеграми хвилинами, і клієнт чекав до тайм-ауту. REST-відповідь — **`202 Accepted`**: «прийняв, виконую», адреса задачі — в `Location`.

```text
клієнт ── POST /api/v1/imports ──────────► API ── запускає фонову задачу
клієнт ◄── 202 + Location: /imports/7f3a ── API
клієнт ── GET /imports/7f3a ─────────────► {"status": "running"}
клієнт ── GET /imports/7f3a ─────────────► {"status": "done", "added": 1}
```

### Приклад 6.1. Запустити імпорт і дочекатися

In [ ]:
def wait_for_job(status_url, pause=0.1, limit=50):
    """Перевіряти стан задачі, поки вона не закінчиться; повертає останній стан."""
    for _ in range(limit):
        state = requests.get(status_url, timeout=5).json()
        if state["status"] in ("done", "failed"):
            return state
        time.sleep(pause)
    raise TimeoutError(f"{status_url}: задача не закінчилась")

In [ ]:
csv_line = "34504,2024,09,02,18,00,AAXX 02181 34504 32975 51106 10251 20129 39989 40151 52027 80001 333 10330="
job = requests.post(f"{BASE}/api/v1/imports", json={"csv": csv_line + "\n99999,2024,09,02,18,00,AAXX ="}, timeout=5)
print(job.status_code, job.json(), "| Location:", job.headers["Location"])
state = wait_for_job(BASE + job.headers["Location"])
print(state)

Відповідь прийшла одразу (`queued`), а робота йшла у фоні. Рядок станції 34504 додано (вже існуюче спостереження просто перезаписалось — дубля немає), рядок неіснуючої станції 99999 з порожньою телеграмою — пропущено. Тут фонова задача живе в тому самому процесі (`BackgroundTasks` FastAPI); у production — черга Redis + Celery (уроки 49–50).

### Приклад 6.2. Неіснуюча задача

In [ ]:
response = requests.get(f"{BASE}/api/v1/imports/nope1234", timeout=5)
print(response.status_code, response.json())

---
## 7. Клієнт до API

`MeteoClient` (файл `weather_map/client.py`) — той самий шаблон, що `SmachnoClient` в уроці 32: адреса, тайм-аут, пагінація й перетворення 404 на `StationNotFound` — в одному місці. Streamlit-карта (`weather_map/app.py`) не містить жодного URL — лише виклики клієнта.

```text
знімок CSV ─► MeteoRepository ─► Meteo API v2 (FastAPI) ─ JSON по HTTP ─► MeteoClient ─┬─► карта Streamlit
(storage.py)                       (app.py)                              (client.py)   └─► ноутбук, pandas
```

### Приклад 7.1. Станції й спостереження через клієнт

In [ ]:
from client import MeteoClient, StationNotFound

client = MeteoClient(BASE)
print([station["name"] for station in client.stations()])
items = client.observations("99001", fields="time,temperature")
print(len(items), items[:2])
for key, value in client.last_exchange.items():
    if key != "response_headers":
        print(f"{key:12} {value}")

`last_exchange` — «HTTP-інспектор» клієнта: який запит пішов, з яким кодом і за скільки мілісекунд. Ним користується вкладка інспектора на карті погоди.

### Приклад 7.2. 404 → `StationNotFound`

In [ ]:
try:
    client.observations("00000")
except StationNotFound as error:
    print("StationNotFound:", error)
print(client.last_exchange["status"])

### Приклад 7.3. Середня температура станції

Код аналізу нічого не знає про HTTP: лише методи клієнта і його виняток.

In [ ]:
def mean_temperature(client, wmo):
    try:
        items = client.observations(wmo, fields="temperature")
    except StationNotFound:
        return None
    return round(sum(item["temperature"] for item in items) / len(items), 1)

In [ ]:
for wmo in ["99001", "34504", "00000"]:
    print(wmo, mean_temperature(client, wmo))
assert mean_temperature(client, "99001") == 15.1

### Приклад 7.4. Середньодобова температура в pandas

Клієнт викладача рахував середньодобові значення в pandas. Зробимо так само — сторінки гортає клієнт, з сервера йдуть лише потрібні поля.

In [ ]:
import pandas as pd

observations = pd.DataFrame(client.observations("99001", fields="time,temperature"))
observations["time"] = pd.to_datetime(observations["time"])
daily = observations.set_index("time")["temperature"].resample("D").agg(["mean", "min", "max", "count"])
print(daily.round(1))

`resample("D")` групує строки по днях (бонусний урок pandas); `count` показує, скільки строків було за добу.

### Задача 2. Фільтр за строком у клієнті

`HourClient` — нащадок `MeteoClient`. Допиши його метод `observations(wmo, fields=..., hour=None)`:

- як у `MeteoClient.observations` — гортає сторінки (`limit=200`, `offset`) через `self._get(...)` і повертає список усіх елементів;
- якщо `hour` задано, передає його **серверу** параметром `hour` (фільтрує сервер, а не Python);
- без `hour` поведінка не змінюється.

Підказка: `requests` сам відкидає параметри зі значенням `None`. Перевірка — у наступній клітинці.

In [ ]:
class HourClient(MeteoClient):
    def observations(self, wmo, fields="time,temperature,pressure,wind_speed", hour=None):
        # BEGIN SOLUTION
        items, offset = [], 0
        while True:
            page = self._get(f"/api/v1/stations/{wmo}/observations", fields=fields, hour=hour,
                             limit=200, offset=offset)
            items += page["items"]
            if page["next"] is None:
                return items
            offset += page["limit"]
        # END SOLUTION

In [ ]:
hour_client = HourClient(BASE)
noon = hour_client.observations("99001", fields="time,temperature", hour=12)
assert noon is not None, "метод має повертати список спостережень"
assert noon == [{"time": "2026-09-20T12:00Z", "temperature": 20.4}], noon
assert "hour=12" in hour_client.last_exchange["request"], "фільтрувати має сервер: hour=12 — у запиті"
everything = hour_client.observations("99001", fields="time")
assert len(everything) == 8 and "hour" not in hour_client.last_exchange["request"]
print(hour_client.last_exchange["request"])
print("✅ Задача 2 розв'язана")

---
## 8. Той самий API без мережі: `TestClient`

FastAPI має `TestClient`: він передає запит у застосунок прямо в пам'яті, без сервера й порту. Так написані тести уроку (`tests/test_api.py`): кожен тест отримує **свіжий** застосунок на фікстурі з однією справжньою телеграмою, тож наші зміни на сервері його не зачіпають.

### Приклад 8.1. Свіжий застосунок і ті самі запити

In [ ]:
from fastapi.testclient import TestClient

from meteo_api.app import create_app
from meteo_api.storage import DATA_DIR, MeteoRepository

fresh_repo = MeteoRepository.from_files(DATA_DIR / "stations_sample.csv", DATA_DIR / "synop_sample.csv")
test = TestClient(create_app(fresh_repo, with_api_types=False))
print(test.get("/api/v1/stations").json()["total"], "станція у свіжому застосунку")
print(test.get("/api/v1/stations/34504/observations/2024-09-02T18:00Z").json()["temperature"])
print(test.get("/api/v1/stations/99001").status_code, "— «Навчальної» тут немає")
print(test.post("/api/v1/stations/34504/observations", json={"time": "2024-09-02T18:00Z", "temperature": 1}).status_code)

Той самий інтерфейс, що в `requests` (`.get`, `.post`, `.json()`), — але без мережі. Тест швидкий і не залежить від того, чи вільний порт.

---
## 9. Знайди помилку

### Приклад 9.1. Ендпоінт, що бреше кодом 200

Колега додав «середню температуру станції»:

In [ ]:
from fastapi import FastAPI, HTTPException

from meteo_api.storage import NotFoundError

buggy = FastAPI()


@buggy.post("/get_mean_temperature")
def get_mean_temperature(wmo: str):
    try:
        items = repo.list_observations(wmo)
        return {"mean": sum(o["temperature"] for o in items) / len(items)}
    except Exception as e:
        return {"error": str(e)}

In [ ]:
buggy_test = TestClient(buggy)
for wmo in ["99001", "00000"]:
    response = buggy_test.post("/get_mean_temperature", params={"wmo": wmo})
    print(wmo, response.status_code, response.json())

Для неіснуючої станції — `200 {"error": …}`: `raise_for_status()` клієнта мовчить, і клієнт падає пізніше з `KeyError: 'mean'`. Проблем кілька:

- **дієслово в URL і `POST` для читання** — має бути `GET` до ресурсу;
- **`except Exception`** ловить усе, навіть помилки в нашому коді, і видає їх за «успіх»;
- **станція без спостережень** дала б `ZeroDivisionError`, теж замаскований під 200.

### Задача 3. Правильна версія ендпоінта

Додай у застосунок `fixed_app` ендпоінт `GET /api/v1/stations/{wmo}/temperature/mean`:

- неіснуюча станція → `404` (лови саме `NotFoundError` зі сховища, а не `Exception`);
- станція без спостережень → `404` з поясненням;
- інакше `200 {"station": wmo, "mean": …}`, середнє округлене до 0.1.

Дані бери з того самого `repo`. Перевірка — у наступній клітинці.

In [ ]:
fixed_app = FastAPI()

# BEGIN SOLUTION
@fixed_app.get("/api/v1/stations/{wmo}/temperature/mean")
def mean_temperature_endpoint(wmo: str):
    try:
        items = repo.list_observations(wmo)
    except NotFoundError as error:
        raise HTTPException(404, str(error))
    if not items:
        raise HTTPException(404, f"у станції {wmo} немає спостережень")
    return {"station": wmo, "mean": round(sum(o["temperature"] for o in items) / len(items), 1)}
# END SOLUTION

In [ ]:
if "99002" not in {station["wmo"] for station in repo.list_stations()}:
    repo.add_station("99002", "Порожня")
fixed_test = TestClient(fixed_app)
ok = fixed_test.get("/api/v1/stations/99001/temperature/mean")
assert ok.status_code == 200, f"для 99001 очікували 200, отримали {ok.status_code}"
assert ok.json() == {"station": "99001", "mean": 15.1}, ok.json()
missing = fixed_test.get("/api/v1/stations/00000/temperature/mean")
assert missing.status_code == 404, "неіснуюча станція — 404"
empty = fixed_test.get("/api/v1/stations/99002/temperature/mean")
assert empty.status_code == 404, "станція без спостережень — 404, а не 500 чи 200"
print(ok.json(), "|", missing.json(), "|", empty.json())
print("✅ Задача 3 розв'язана")

---
## 10. Зупиняємо сервери

### Приклад 10.1. Зупинити uvicorn і gRPC

Сервери працювали у фонових потоках ядра. Зупиняємо їх і звільняємо порти. Щоб запустити знову — перезапусти ядро і клітинки зверху.

In [ ]:
if meteo_app._server is not None:
    meteo_app._server.should_exit = True          # uvicorn закінчить поточні запити і зупиниться
    for _ in range(50):
        try:
            requests.get(f"{BASE}/api/v1/stations", timeout=0.5)
        except requests.ConnectionError:
            break
        time.sleep(0.1)
    meteo_app._server = None
if grpc is not None and grpc_meteo._server is not None:
    grpc_meteo._server.stop(0)
    grpc_meteo._server = None
try:
    requests.get(f"{BASE}/api/v1/stations", timeout=1)
except requests.ConnectionError:
    print("Meteo API зупинено:", BASE)

---
## Самоперевірка

1. Чому пошук — це `GET` з параметрами, а не `POST /filter…`?
2. Які коди означають: створено; вже є; дані неправильні; не знайдено; метод не підтримується; прийнято, виконується?
3. Навіщо в конверті сторінки `next`?
4. Чим `PATCH` відрізняється від `PUT`?
5. Чим GraphQL і JSON-RPC відрізняються від REST у тому, як повідомляють про помилки?
6. Коли обирають gRPC, а коли webhook?

<details>
<summary>Відповіді</summary>

1. Пошук лише читає: `GET` ідемпотентний, кешується, його можна відкрити в браузері; дієслово в URL суперечить ідеї ресурсів.
2. `201` (+ `Location`), `409`, `422`, `404`, `405` (+ `Allow`), `202` (+ `Location`).
3. Клієнт іде за готовим посиланням і не рахує `offset` сам; сервер може змінити спосіб пагінації.
4. `PATCH` змінює лише передані поля, `PUT` замінює ресурс цілком.
5. Вони відповідають `200` і кладуть помилку в тіло (`error` / `errors`); REST повідомляє про помилку статус-кодом.
6. gRPC — швидкий зв'язок своїх сервісів між собою за суворим контрактом; webhook — коли сервер має сам повідомити інший сервер про подію (оплата, новий коміт).

</details>

## Далі

- Книга: [Урок 33](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_33/) — усі типи API з прикладами, `202 Accepted`, OpenAPI, архітектура й карта погоди на Streamlit.
- Карта: `uvicorn meteo_api.app:app --port 8032` і `streamlit run weather_map/app.py` у папці уроку.
- **Урок 34** — Django intro: MVT, ORM, admin.